# Production Particle Tracking - Google Colab Version

**Fully-Fused RK4 with Time-Dependent Velocity**

This notebook runs JAXTrace particle tracking on Google Colab T4 GPU.

## Features:
- Cyclic velocity sequence (40 timesteps, wraps periodically)
- All velocity fields pre-loaded on GPU
- Single vmap over particles (all RK4 stages fused)
- NO CPU-GPU transfers between timesteps
- L0 → L1 → L2 hierarchical search (Morton or Hilbert curves)

## Requirements:
- Google Colab with T4 GPU (free tier)
- PVTU mesh data in Google Drive
- ~16 GB GPU memory recommended

## 1. Setup: Install JAX and Dependencies

In [ ]:
%%bash
# Check GPU availability
nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%bash
# Install JAX for CUDA 12 (T4 GPU)
pip install -q --upgrade pip
pip install -q --upgrade "jax[cuda12]"

# Install other dependencies
pip install -q numpy scipy vtk meshio

echo "✅ Installation complete"

In [ ]:
# Verify JAX GPU setup
import jax
import jax.numpy as jnp

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"Default backend: {jax.default_backend()}")

# Test GPU
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print(f"\n✅ JAX GPU test passed: {y.shape}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted at /content/drive")

## 3. Setup JAXTrace from Google Drive

**⚠️ IMPORTANT: Complete this before running the next cell!**

### Required Setup:

1. **Upload JAXTrace directory to Google Drive**
   - Upload the entire `JAXTrace` folder to: `/MyDrive/JAXTrace/`
   - The directory must contain the `jaxtrace` subdirectory with all modules

2. **Upload mesh data to Google Drive**
   - Upload your PVTU mesh files to: `/MyDrive/welding_data/FLA/post/0eule/`
   - Or adjust the path in the Configuration cell below

### Expected Google Drive Structure:

```
/content/drive/MyDrive/
├── JAXTrace/                      ← Upload this directory
│   ├── jaxtrace/                  ← Python module (required!)
│   │   ├── __init__.py
│   │   ├── gpu/
│   │   ├── tracking/
│   │   ├── io/
│   │   └── ...
│   ├── production_tracking_fully_fused_timedep.py
│   └── ...
└── welding_data/                  ← Your mesh data
    └── FLA/post/0eule/
        ├── featurelessAvtk_120.pvtu
        ├── featurelessAvtk_121.pvtu
        └── ...
```

### What the next cell does:

- Verifies `JAXTrace` directory exists at `/content/drive/MyDrive/JAXTrace/`
- Checks for required subdirectories (`jaxtrace/`, `jaxtrace/gpu/`)
- Adds JAXTrace to Python's import path
- Tests that `import jaxtrace` works correctly

In [ ]:
import sys
import os
from pathlib import Path

# ============================================================================
# Path to JAXTrace in your Google Drive
# ============================================================================
JAXTRACE_PATH = Path("/content/drive/MyDrive/JAXTrace")

# Verify JAXTrace exists
if not JAXTRACE_PATH.exists():
    raise FileNotFoundError(
        f"\n❌ JAXTrace not found at {JAXTRACE_PATH}\n"
        f"   Please upload the JAXTrace directory to your Google Drive.\n"
        f"   Expected structure:\n"
        f"     /content/drive/MyDrive/JAXTrace/\n"
        f"     ├── jaxtrace/\n"
        f"     ├── production_tracking_fully_fused_timedep.py\n"
        f"     └── ..."
    )

# Check for key directories
jaxtrace_module = JAXTRACE_PATH / "jaxtrace"
if not jaxtrace_module.exists():
    raise FileNotFoundError(
        f"\n❌ jaxtrace module not found at {jaxtrace_module}\n"
        f"   Make sure you uploaded the complete JAXTrace directory.\n"
        f"   The 'jaxtrace' subdirectory must be present."
    )

# Check for GPU subdirectory
gpu_module = jaxtrace_module / "gpu"
if not gpu_module.exists():
    raise FileNotFoundError(
        f"\n❌ jaxtrace.gpu module not found at {gpu_module}\n"
        f"   Incomplete JAXTrace directory structure."
    )

# Add to Python path
jaxtrace_path_str = str(JAXTRACE_PATH)
if jaxtrace_path_str not in sys.path:
    sys.path.insert(0, jaxtrace_path_str)

print(f"✅ JAXTrace found at: {JAXTRACE_PATH}")
print(f"✅ Added to Python path: {jaxtrace_path_str}")

# Verify import works
try:
    import jaxtrace
    jaxtrace_location = jaxtrace.__file__
    print(f"✅ jaxtrace module imported successfully")
    print(f"   Module location: {jaxtrace_location}")
except ImportError as e:
    print(f"\n❌ Failed to import jaxtrace: {e}")
    print(f"   Python path: {sys.path}")
    raise

## 4. Configuration

In [ ]:
import os
from pathlib import Path

# ============================================================================
# MESH DATA CONFIGURATION (Google Drive paths)
# ============================================================================

# Path to your mesh data in Google Drive
# Example: /content/drive/MyDrive/welding_data/FLA/post/0eule
MESH_BASE_PATH = Path("/content/drive/MyDrive/welding_data/FLA/post/0eule")

MESH_FILE_PATTERN = "featurelessAvtk_{timestep}.pvtu"
VELOCITY_TIMESTEP_RANGE = (120, 159)  # Load timesteps 120-159 (40 timesteps)
VELOCITY_FIELD_NAME = 'Displacement'  # Field name in PVTU files
VELOCITY_DT = 0.0025  # Time spacing between velocity snapshots

# ============================================================================
# PARTICLE CONFIGURATION
# ============================================================================

# Reduced for Colab T4 GPU (15 GB memory)
PARTICLE_GRID_RESOLUTION = (30, 60, 30)  # 54,000 particles (vs 225,000 on workstation)

PARTICLE_BOUNDS_FRACTION = {
    'x': (0.2, 0.35),  # Use first 20% of domain in X (entrance region)
    'y': (0.2, 0.8),   # Full domain in Y
    'z': (0.3, 1.0),   # Full domain in Z
}

N_X, N_Y, N_Z = PARTICLE_GRID_RESOLUTION
N_PARTICLES = N_X * N_Y * N_Z

# ============================================================================
# TRACKING CONFIGURATION
# ============================================================================

DT = 0.0025
N_STEPS = 1_000  # Reduced for Colab (vs 2,500 on workstation)

# Space-filling curve selection
CURVE_TYPE = 'morton'  # 'morton' or 'hilbert'

# Neighbor method
NEIGHBOR_METHOD = 'face'  # 'face' or 'node'

# L2 search method
L2_SEARCH_METHOD = 'neighbors'  # 'radius', 'neighbors', or 'hierarchical'

N_HOPS = 3
L2_SEARCH_RADIUS = 10
ENABLE_L1_SEARCH = True

# Initial assignment radii (curve-dependent)
if CURVE_TYPE == 'hilbert':
    INITIAL_SEARCH_RADIUS = 75
    INITIAL_SEARCH_FALLBACK_RADII = [150, 300, 600]
else:  # morton
    INITIAL_SEARCH_RADIUS = 50
    INITIAL_SEARCH_FALLBACK_RADII = [100, 200, 300]

SEED = 42
LOG_INTERVAL = 50  # More frequent logging for Colab

# ============================================================================
# EXPORT CONFIGURATION
# ============================================================================

EXPORT_FREQUENCY = 20  # Export every 20 timesteps
OUTPUT_DIR = Path("/content/output")  # Local Colab storage (faster than Drive)
STORE_VELOCITIES = False

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("Configuration Summary")
print("=" * 80)
print(f"Mesh path: {MESH_BASE_PATH}")
print(f"Particles: {N_PARTICLES:,} ({N_X} × {N_Y} × {N_Z})")
print(f"Timesteps: {N_STEPS:,}")
print(f"Curve type: {CURVE_TYPE}")
print(f"Output: {OUTPUT_DIR}")
print("=" * 80)

## 5. JAX Memory Configuration

In [ ]:
# Configure JAX for efficient GPU memory usage
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

print("✅ JAX memory configuration set")

## 6. Import JAXTrace Modules

In [ ]:
import time
import queue
import threading
import numpy as np
import jax
import jax.numpy as jnp
from dataclasses import dataclass
from typing import Dict

from jaxtrace.gpu.particles import ParticleData
from jaxtrace.gpu.mesh_loader_timedep import load_velocity_sequence_from_pvtu, compute_velocity_cycle_params
from jaxtrace.gpu.tracking.mesh_data_gpu import upload_mesh_to_gpu
from jaxtrace.gpu.forest import build_element_neighbors_array
from jaxtrace.gpu.search.morton_octree_builder import build_global_morton_octree
from jaxtrace.gpu.search.hilbert_octree_builder import build_global_hilbert_octree
from jaxtrace.gpu.search.morton_global_search import upload_global_morton_to_gpu
from jaxtrace.gpu.tracking.rk4_fully_fused_timedep import create_rk4_fully_fused_timedep
from jaxtrace.gpu.tracking.initial_assignment_cascading import initial_assignment_cascading_fallback
from jaxtrace.tracking.seeding import uniform_grid_seeds

print("✅ All modules imported successfully")

## 7. Async VTK Exporter (Helper Class)

In [ ]:
@dataclass
class ExportConfig:
    """Configuration for VTK export"""
    output_dir: Path
    export_frequency: int
    include_velocities: bool = True
    include_metadata: bool = True


class AsyncVTKExporter:
    """
    Async VTK exporter that runs in background thread.
    
    Minimal memory overhead: Only stores current timestep data in queue.
    """

    def __init__(self, config: ExportConfig, particle_data_template: ParticleData):
        self.config = config
        self.template = particle_data_template
        self.export_queue = queue.Queue(maxsize=5)
        self.worker_thread = None
        self.stop_event = threading.Event()
        self.n_exported = 0
        self.export_times = []

        self.config.output_dir.mkdir(parents=True, exist_ok=True)

    def start(self):
        """Start background export worker"""
        self.worker_thread = threading.Thread(target=self._export_worker, daemon=True)
        self.worker_thread.start()

    def _export_worker(self):
        """Background thread that processes export queue"""
        while not self.stop_event.is_set():
            try:
                export_data = self.export_queue.get(timeout=1.0)

                if export_data is None:
                    break

                step, positions, velocities, element_ids, active_mask = export_data

                t0 = time.perf_counter()
                output_file = self.config.output_dir / f"particles_step_{step:06d}.vtu"

                active_positions = positions[active_mask]
                active_velocities = velocities[active_mask] if (velocities is not None and self.config.include_velocities) else None

                from jaxtrace.io import VTKTrajectoryWriter
                writer = VTKTrajectoryWriter()
                writer.write_particles_at_time(
                    positions=active_positions,
                    velocities=active_velocities,
                    time=step,
                    filename=str(output_file),
                    format='xml'
                )

                export_time = time.perf_counter() - t0
                self.export_times.append(export_time)
                self.n_exported += 1
                self.export_queue.task_done()

            except queue.Empty:
                continue
            except Exception as e:
                print(f"Export error: {e}")

    def enqueue_export(self, step: int, particle_data: ParticleData):
        """Add particle data to export queue (non-blocking)"""
        try:
            positions = np.array(particle_data.positions, dtype=np.float32)

            if self.config.include_velocities:
                velocities = np.array(particle_data.velocities, dtype=np.float32)
            else:
                velocities = None

            element_ids = np.array(particle_data.element_ids, dtype=np.int32)
            active_mask = np.array(particle_data.active_mask, dtype=bool)

            self.export_queue.put(
                (step, positions, velocities, element_ids, active_mask),
                timeout=10.0
            )
        except queue.Full:
            print(f"Warning: Export queue full at step {step}, skipping export")

    def stop(self):
        """Stop background worker and wait for queue to finish"""
        self.export_queue.put(None)
        self.stop_event.set()

        if self.worker_thread:
            self.worker_thread.join(timeout=30.0)

    def get_stats(self) -> Dict:
        """Get export statistics"""
        if not self.export_times:
            return {'n_exported': 0, 'mean_time': 0, 'total_time': 0}

        return {
            'n_exported': self.n_exported,
            'mean_time': np.mean(self.export_times),
            'total_time': np.sum(self.export_times),
            'queue_size': self.export_queue.qsize(),
        }

print("✅ Async VTK exporter defined")

## 8. Load Mesh and Velocity Sequence

In [ ]:
print("=" * 80)
print("Loading Mesh and Velocity Sequence")
print("=" * 80)

t_load = time.time()
node_positions, connectivity, velocity_sequence = load_velocity_sequence_from_pvtu(
    base_path=MESH_BASE_PATH,
    file_pattern=MESH_FILE_PATTERN,
    timestep_range=VELOCITY_TIMESTEP_RANGE,
    field_name=VELOCITY_FIELD_NAME,
    verbose=True
)
t_load = time.time() - t_load

n_nodes = node_positions.shape[0]
n_elements = connectivity.shape[0]
n_velocity_steps = velocity_sequence.shape[0]

print(f"\nMesh: {n_elements:,} elements, {n_nodes:,} nodes")
print(f"Velocity timesteps: {n_velocity_steps}")
print(f"Total load time: {t_load:.2f}s")

# Compute velocity cycle parameters
cycle_params = compute_velocity_cycle_params(
    total_steps=N_STEPS,
    dt=DT,
    velocity_timestep_range=VELOCITY_TIMESTEP_RANGE,
    velocity_dt=VELOCITY_DT
)
print(f"\nVelocity cycle parameters:")
print(f"  Cycle period: {cycle_params['cycle_period']:.3f} time units")
print(f"  Number of cycles: {cycle_params['n_cycles']:.2f}")
print(f"  Tracking steps per velocity step: {cycle_params['steps_per_velocity']}")

## 9. Build Space-Filling Curve Octree

In [ ]:
print("\n" + "=" * 80)
print(f"Building Global {CURVE_TYPE.upper()} Structure")
print("=" * 80)

t_octree = time.time()

if CURVE_TYPE == 'hilbert':
    octree_struct = build_global_hilbert_octree(
        node_positions=node_positions,
        connectivity=connectivity,
        leaf_capacity=256,
        max_depth=21,
        verbose=False
    )
    curve_field_name = 'hilbert_sorted'
elif CURVE_TYPE == 'morton':
    octree_struct = build_global_morton_octree(
        node_positions=node_positions,
        connectivity=connectivity,
        leaf_capacity=256,
        max_depth=21,
        verbose=False
    )
    curve_field_name = 'morton_sorted'
else:
    raise ValueError(f"Unknown CURVE_TYPE: {CURVE_TYPE}")

t_octree = time.time() - t_octree

curve_indices = getattr(octree_struct, curve_field_name)
octree_memory_mb = (octree_struct.elem_ids_sorted.nbytes + curve_indices.nbytes) / (1024**2)

print(f"Built {octree_struct.n_leaves:,} leaves in {t_octree:.2f}s")
print(f"Memory: {octree_memory_mb:.1f} MB")
print(f"Prefix table depth: {octree_struct.table_depth}")

## 10. Upload to GPU

In [ ]:
print("\n" + "=" * 80)
print("Uploading to GPU")
print("=" * 80)

t_upload = time.time()

# Compute element neighbors
neighbor_method_name = "NODE-BASED" if NEIGHBOR_METHOD == 'node' else "FACE-BASED"
print(f"\nComputing element neighbors ({neighbor_method_name})...")
t_neighbors = time.time()
element_neighbors = build_element_neighbors_array(connectivity, method=NEIGHBOR_METHOD, verbose=True)
t_neighbors = time.time() - t_neighbors
print(f"  Neighbor computation: {t_neighbors:.2f}s")
neighbor_memory_mb = element_neighbors.nbytes / (1024**2)
print(f"  Neighbor memory: {neighbor_memory_mb:.1f} MB")

# Upload mesh data
mesh_gpu = upload_mesh_to_gpu(
    connectivity=connectivity,
    node_positions=node_positions,
    element_neighbors=element_neighbors,
    verbose=False
)

# Compute element volumes for adaptive L1
print("\nComputing element volumes...")
t_volumes = time.time()
v0 = node_positions[connectivity[:, 0]]
v1 = node_positions[connectivity[:, 1]]
v2 = node_positions[connectivity[:, 2]]
v3 = node_positions[connectivity[:, 3]]
e1 = v1 - v0
e2 = v2 - v0
e3 = v3 - v0
cross_e2_e3 = np.cross(e2, e3)
det = np.sum(e1 * cross_e2_e3, axis=1)
element_volumes_cpu = np.abs(det) / 6.0
element_volumes_gpu = jax.device_put(element_volumes_cpu.astype(np.float32))
t_volumes = time.time() - t_volumes
print(f"  Volume range: [{element_volumes_cpu.min():.2e}, {element_volumes_cpu.max():.2e}]")
print(f"  Computation time: {t_volumes:.2f}s")

# Upload octree structure
mesh_gpu_octree = upload_global_morton_to_gpu(
    octree_struct,
    connectivity,
    node_positions
)

# Force transfer
_ = jax.block_until_ready(mesh_gpu.connectivity)
_ = jax.block_until_ready(mesh_gpu_octree.elem_ids_sorted)

t_upload = time.time() - t_upload
print(f"\nTotal upload time: {t_upload:.2f}s")
print(f"{CURVE_TYPE.upper()} GPU leaves: {mesh_gpu_octree.n_leaves:,}")
print(f"{CURVE_TYPE.upper()} Prefix Table Depth: {mesh_gpu_octree.table_depth}")

## 11. Initialize Particles

In [ ]:
print("\n" + "=" * 80)
print(f"Initializing {N_PARTICLES:,} Particles")
print("=" * 80)

# Compute domain bounds
domain_min = node_positions.min(axis=0)
domain_max = node_positions.max(axis=0)
domain_size = domain_max - domain_min

# Compute particle bounds
par_bounds_min = np.zeros(3, dtype=np.float32)
par_bounds_max = np.zeros(3, dtype=np.float32)
for i, axis in enumerate(['x', 'y', 'z']):
    min_frac, max_frac = PARTICLE_BOUNDS_FRACTION[axis]
    par_bounds_min[i] = domain_min[i] + min_frac * domain_size[i]
    par_bounds_max[i] = domain_min[i] + max_frac * domain_size[i]
par_bounds = [par_bounds_min, par_bounds_max]

print(f"\nParticle bounds:")
print(f"  X: [{par_bounds_min[0]:.6f}, {par_bounds_max[0]:.6f}]")
print(f"  Y: [{par_bounds_min[1]:.6f}, {par_bounds_max[1]:.6f}]")
print(f"  Z: [{par_bounds_min[2]:.6f}, {par_bounds_max[2]:.6f}]")

# Generate uniform grid
particle_positions = uniform_grid_seeds(
    resolution=(N_X, N_Y, N_Z),
    bounds=par_bounds,
    include_boundaries=True
)

# Clip to mesh bounds
print(f"\nClipping particles to mesh bounds...")
margin = 0.01
bbox_min_safe = domain_min + margin * (domain_max - domain_min)
bbox_max_safe = domain_max - margin * (domain_max - domain_min)
particle_positions = np.clip(particle_positions, bbox_min_safe, bbox_max_safe)

# Create particle data
particle_data = ParticleData.from_positions(particle_positions)

print(f"\n✅ Created {N_PARTICLES:,} particles")

## 12. Setup Async Export

In [ ]:
print("\n" + "=" * 80)
print("Setting up Async VTK Export")
print("=" * 80)

export_config = ExportConfig(
    output_dir=OUTPUT_DIR,
    export_frequency=EXPORT_FREQUENCY,
    include_velocities=STORE_VELOCITIES,
    include_metadata=True
)

exporter = AsyncVTKExporter(export_config, particle_data)
exporter.start()

print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"Export frequency: every {EXPORT_FREQUENCY} steps")
print(f"Expected exports: {N_STEPS // EXPORT_FREQUENCY}")

## 13. Run Time Integration

In [ ]:
print("\n" + "=" * 80)
print(f"Running Time Integration ({N_STEPS:,} steps)")
print("=" * 80)

print(f"\nSearch hierarchy configuration:")
if ENABLE_L1_SEARCH:
    if L2_SEARCH_METHOD == 'neighbors':
        print(f"  L0 (cached) → L1 ({N_HOPS} hops) → L2 ({CURVE_TYPE.upper()} neighbors, 27 octants)")
    else:
        print(f"  L0 (cached) → L1 ({N_HOPS} hops) → L2 ({CURVE_TYPE.upper()} radius, ±{L2_SEARCH_RADIUS})")
else:
    print(f"  L0 (cached) → L2 ({CURVE_TYPE.upper()})")

print(f"  L2 method: {L2_SEARCH_METHOD}")

# Create RK4 step function
rk4_step = create_rk4_fully_fused_timedep(
    mesh_gpu_connectivity=mesh_gpu.connectivity,
    mesh_gpu_node_positions=mesh_gpu.node_positions,
    mesh_gpu_element_neighbors=mesh_gpu.element_neighbors,
    mesh_gpu_element_volumes=element_volumes_gpu,
    mesh_gpu_global_morton=mesh_gpu_octree,
    n_hops=N_HOPS,
    l2_search_radius=L2_SEARCH_RADIUS,
    enable_l1_search=ENABLE_L1_SEARCH,
    l2_search_method=L2_SEARCH_METHOD
)

# Upload data to GPU
print("\nUploading data to GPU...")
velocity_fields_gpu = jax.device_put(velocity_sequence)
positions_gpu = jax.device_put(particle_data.positions)
element_ids_gpu = jax.device_put(particle_data.element_ids)

# Initial assignment
print(f"\nRunning cascading initial assignment...")
print(f"  Initial radius: {INITIAL_SEARCH_RADIUS}")
print(f"  Fallback radii: {INITIAL_SEARCH_FALLBACK_RADII}")
t_initial = time.time()
element_ids_gpu = initial_assignment_cascading_fallback(
    positions_gpu,
    mesh_gpu_octree,
    initial_radius=INITIAL_SEARCH_RADIUS,
    fallback_radii=INITIAL_SEARCH_FALLBACK_RADII,
    verbose=True
)
element_ids_gpu = jax.block_until_ready(element_ids_gpu)
t_initial = time.time() - t_initial

n_active_initial = int(jnp.sum(element_ids_gpu >= 0))
initial_success_rate = (n_active_initial / N_PARTICLES) * 100
print(f"  Initial assignment: {n_active_initial:,}/{N_PARTICLES:,} ({initial_success_rate:.2f}%)")
print(f"  Search time: {t_initial:.2f}s")

# Compile (first step)
print("\nCompiling RK4 (first step)...")
t_compile = time.time()
positions_gpu, element_ids_gpu = rk4_step(
    positions_gpu,
    element_ids_gpu,
    DT,
    velocity_fields_gpu,
    0
)
positions_gpu = jax.block_until_ready(positions_gpu)
element_ids_gpu = jax.block_until_ready(element_ids_gpu)
t_compile = time.time() - t_compile
print(f"  Compilation time: {t_compile:.2f}s")

# Main integration loop
print(f"\nRunning {N_STEPS:,} timesteps...")
print(f"{'Step':>6} {'Active':>10} {'Retention':>10} {'Step Time':>12} {'Throughput':>15}")
print(f"{'-'*6} {'-'*10} {'-'*10} {'-'*12} {'-'*15}")

t_integration_start = time.time()
step_times = []
retention_history = []

for step in range(1, N_STEPS + 1):
    t_step = time.time()

    # Run RK4 step
    positions_gpu, element_ids_gpu = rk4_step(
        positions_gpu,
        element_ids_gpu,
        DT,
        velocity_fields_gpu,
        step
    )

    positions_gpu = jax.block_until_ready(positions_gpu)
    element_ids_gpu = jax.block_until_ready(element_ids_gpu)

    t_step = time.time() - t_step
    step_times.append(t_step)

    # Count active particles
    n_active = int(jnp.sum(element_ids_gpu >= 0))
    retention = (n_active / N_PARTICLES) * 100
    retention_history.append(retention)
    throughput = N_PARTICLES / t_step

    # Export
    if step % EXPORT_FREQUENCY == 0:
        positions_cpu = np.array(positions_gpu, dtype=np.float32)
        element_ids_cpu = np.array(element_ids_gpu, dtype=np.int32)
        particle_data_export = ParticleData(
            positions=positions_cpu,
            velocities=np.zeros((N_PARTICLES, 3), dtype=np.float32),
            element_ids=element_ids_cpu,
            block_ids=np.zeros(N_PARTICLES, dtype=np.int32),
            active_mask=(element_ids_cpu >= 0)
        )
        exporter.enqueue_export(step, particle_data_export)

    # Log
    if step % LOG_INTERVAL == 0 or step == N_STEPS:
        export_stats = exporter.get_stats()
        print(f"{step:6d} {n_active:10,} {retention:9.2f}% {t_step*1000:10.2f} ms {throughput:12.0f} p/s | Exported: {export_stats['n_exported']:>4}")

t_integration = time.time() - t_integration_start

print(f"\n✅ Integration complete: {t_integration:.2f}s")

## 14. Finalize Export

In [ ]:
print("\nWaiting for exports to complete...")
exporter.stop()

export_stats = exporter.get_stats()
print(f"✅ All exports complete")
print(f"  Files exported: {export_stats['n_exported']}")
print(f"  Mean export time: {export_stats['mean_time']:.3f} s")
print(f"  Total export time: {export_stats['total_time']:.1f} s")

## 15. Results Summary

In [ ]:
# Download final state
positions_final_cpu = np.array(positions_gpu, dtype=np.float32)
element_ids_final_cpu = np.array(element_ids_gpu, dtype=np.int32)

final_active = np.sum(element_ids_final_cpu >= 0)
final_retention = (final_active / N_PARTICLES) * 100

# Timing statistics
mean_step_time = np.mean(step_times[1:])
std_step_time = np.std(step_times[1:])
min_step_time = np.min(step_times[1:])
max_step_time = np.max(step_times[1:])
mean_throughput = N_PARTICLES / mean_step_time

# Retention milestones
retention_10 = retention_history[9] if len(retention_history) > 9 else 0
retention_100 = retention_history[99] if len(retention_history) > 99 else 0
retention_500 = retention_history[499] if len(retention_history) > 499 else 0

print("\n" + "=" * 80)
print("PRODUCTION RESULTS")
print("=" * 80)

print(f"\nInitial particles: {N_PARTICLES:,}")
print(f"Initial assignment: {n_active_initial:,} ({initial_success_rate:.2f}%)")
print(f"Final active: {final_active:,}")
print(f"Final retention: {final_retention:.2f}%")

print(f"\nTimesteps completed: {N_STEPS:,}")
print(f"Total integration time: {t_integration:.2f}s")
print(f"Mean step time: {mean_step_time*1000:.2f} ± {std_step_time*1000:.2f} ms")
print(f"Min/Max step time: {min_step_time*1000:.2f} / {max_step_time*1000:.2f} ms")
print(f"Mean throughput: {mean_throughput:.0f} particles/s")

print(f"\nRetention history:")
print(f"  Step 10:    {retention_10:.2f}%")
print(f"  Step 100:   {retention_100:.2f}%")
print(f"  Step 500:   {retention_500:.2f}%")
print(f"  Step {N_STEPS}: {final_retention:.2f}%")

print("\n" + "=" * 80)
print("PERFORMANCE METRICS")
print("=" * 80)

success = True

if initial_success_rate >= 95.0:
    print(f"✅ Initial assignment: {initial_success_rate:.2f}% (≥95% target)")
else:
    print(f"❌ Initial assignment: {initial_success_rate:.2f}% (<95% target)")
    success = False

if final_retention >= 95.0:
    print(f"✅ Final retention: {final_retention:.2f}% (≥95% target)")
else:
    print(f"❌ Final retention: {final_retention:.2f}% (<95% target)")
    success = False

if mean_throughput >= 20000:
    print(f"✅ Throughput: {mean_throughput:.0f} p/s (≥20k target for Colab)")
else:
    print(f"⚠️  Throughput: {mean_throughput:.0f} p/s (<20k target)")

print(f"✅ Architecture: L0 (cached) + L1 ({N_HOPS}-hop) + L2 ({CURVE_TYPE.upper()}, {L2_SEARCH_METHOD})")
print(f"✅ {CURVE_TYPE.upper()} structure: {octree_struct.n_leaves:,} leaves")
print(f"✅ VTK export: {export_stats['n_exported']} files in {OUTPUT_DIR}")

print("=" * 80)

if success:
    print("\n🎉 PRODUCTION TEST PASSED!")
else:
    print("\n⚠️  PRODUCTION TEST RESULTS")
    print("   Some metrics below target.")

print("=" * 80)

## 16. Plot Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Retention over time
axes[0].plot(range(1, len(retention_history) + 1), retention_history, 'b-', linewidth=2)
axes[0].axhline(y=95, color='r', linestyle='--', label='95% target')
axes[0].set_xlabel('Timestep', fontsize=12)
axes[0].set_ylabel('Retention (%)', fontsize=12)
axes[0].set_title('Particle Retention Over Time', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Step time distribution
axes[1].hist(np.array(step_times[1:]) * 1000, bins=50, color='green', alpha=0.7, edgecolor='black')
axes[1].axvline(x=mean_step_time * 1000, color='r', linestyle='--', linewidth=2, label=f'Mean: {mean_step_time*1000:.2f} ms')
axes[1].set_xlabel('Step Time (ms)', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Step Time Distribution', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'tracking_results.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Plot saved to {OUTPUT_DIR / 'tracking_results.png'}")

## 17. Copy Results to Google Drive (Optional)

In [ ]:
# Copy output files to Google Drive for permanent storage
import shutil

drive_output_dir = Path("/content/drive/MyDrive/jaxtrace_results")
drive_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Copying results to Google Drive: {drive_output_dir}")
print("This may take several minutes...")

# Copy VTK files
vtk_files = list(OUTPUT_DIR.glob("*.vtu"))
for i, vtk_file in enumerate(vtk_files, 1):
    shutil.copy2(vtk_file, drive_output_dir / vtk_file.name)
    if i % 10 == 0:
        print(f"  Copied {i}/{len(vtk_files)} files...")

# Copy plot
plot_file = OUTPUT_DIR / 'tracking_results.png'
if plot_file.exists():
    shutil.copy2(plot_file, drive_output_dir / plot_file.name)

print(f"\n✅ Results copied to: {drive_output_dir}")
print(f"   Total files: {len(vtk_files) + 1}")

## 18. Cleanup (Optional)

In [ ]:
# Free GPU memory
del positions_gpu, element_ids_gpu, velocity_fields_gpu
del mesh_gpu, mesh_gpu_octree, element_volumes_gpu
jax.clear_caches()

print("✅ GPU memory cleared")